<a href="https://colab.research.google.com/github/kjahan/armory/blob/main/notebooks/t5_paraphraser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Paraphrase-Generation

Model: `Vamsi/T5_Paraphrase_Paws`

Ref: https://huggingface.co/Vamsi/T5_Paraphrase_Paws


## Install transformers with sentencepiece

We ran into the following issue so we need to install `sentencepiece` to resolve it:

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece installed to convert a slow tokenizer to a fast one.



In [1]:
!pip install transformers[sentencepiece]

     |████████████████████████████████| 3.4 MB 5.6 MB/s 
     |████████████████████████████████| 895 kB 47.7 MB/s 
     |████████████████████████████████| 61 kB 466 kB/s 
     |████████████████████████████████| 596 kB 47.2 MB/s 
     |████████████████████████████████| 3.3 MB 34.9 MB/s 
     |████████████████████████████████| 1.2 MB 40.2 MB/s 
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 3.13
    Uninstalling PyYAML-3.13:
      Successfully uninstalled PyYAML-3.13


## Imports

### Paraphrase-Generation
​

### Model description
​T5 Model for generating paraphrases of english sentences. Trained on the Google PAWS dataset.​

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

## Setup tokenizer/model

In [7]:
tokenizer = AutoTokenizer.from_pretrained("Vamsi/T5_Paraphrase_Paws")  
model = AutoModelForSeq2SeqLM.from_pretrained("Vamsi/T5_Paraphrase_Paws").to("cuda")

## Your input sentence for paraphrasing

In [36]:
# sentence = "The spread of Omicron has roiled financial markets and prompted governments around the world to tighten travel and workplace restrictions."
# sentence = "The Federal Reserve has spent most of 2021 saying that high inflation would be temporary."
sentence = "The good news is that the Fed can taper fast enough to let it raise interest rates in March."

text =  "paraphrase: " + sentence + " </s>"

## Prepare inputs

Make sure you change your `Runtime` to run this notebook on a GPU; otherwise, we will run into the following error:

`RuntimeError: No CUDA GPUs are available`

In [37]:
encoding = tokenizer.encode_plus(text, pad_to_max_length=True, return_tensors="pt")
input_ids, attention_masks = encoding["input_ids"].to("cuda"), encoding["attention_mask"].to("cuda")

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2232: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  FutureWarning,


## Generate output

We had to make sure to load the model in GPU with the following command:

`model = AutoModelForSeq2SeqLM.from_pretrained("Vamsi/T5_Paraphrase_Paws").to("cuda")`

Otherwise, we would run to a RunTime error as shown below:

`RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper__index_select)`

Refrence:
https://discuss.huggingface.co/t/runtimeerror-expected-all-tensors-to-be-on-the-same-device-but-found-at-least-two-devices-cpu-and-cuda-0-when-checking-arugment-for-argument-index-in-method-wrapper-index-select/9255

In [38]:
outputs = model.generate(
    input_ids=input_ids, attention_mask=attention_masks,
    max_length=256,
    do_sample=True,
    top_k=120,
    top_p=0.95,
    early_stopping=True,
    num_return_sequences=5
)

## Print paraphrased output

`"The spread of Omicron has roiled financial markets and prompted governments around the world to tighten travel and workplace restrictions."
`

In [15]:
for output in outputs:
    line = tokenizer.decode(output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print(line)

The spread of Omicron has twisted the financial markets and prompted governments around the world to tighten travel and workplace restrictions.
The spread of Omicron has roiled the financial markets and prompted governments around the world to tighten travel and workplace restrictions.
The spread of Omicron has roiled the financial markets and prompted governments around the world to tighten travel and workplace restrictions.
The spread of Omicron roiled the financial markets and prompted governments around the world to tighten travel restrictions and workplace regulations.
The spread of Omicron has disrupted financial markets and forced governments around the world to tighten travel and workplace restrictions.


## Paraphrased

`sentence = "The Federal Reserve has spent most of 2021 saying that high inflation would be temporary."`


In [31]:
print("Original paragraph:")
print("{}\n".format(sentence))

for output in outputs:
    line = tokenizer.decode(output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print(line)

Original paragraph:
The Federal Reserve has spent most of 2021 saying that high inflation would be temporary.

The Federal Reserve has spent most of 2021 saying that high inflation would be temporary.
The Federal Reserve has spent the majority of 2021 saying that high inflation would be temporary.
The Federal Reserve has spent most of 2021 saying that high inflation would be temporary.
The Federal Reserve has spent most of 2021 declaring that high inflation would be temporary.
The Federal Reserve spent most of 2021 saying that high inflation would be temporary.


## Generate paraphrase

`sentence = "The good news is that the Fed can taper fast enough to let it raise interest rates in March."`

In [39]:
print("Original paragraph:")
print("{}\n".format(sentence))

for output in outputs:
    line = tokenizer.decode(output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print(line)

Original paragraph:
The good news is that the Fed can taper fast enough to let it raise interest rates in March.

Good news is that the Fed can quickly increase interest rates to allow it to raise rates in March.
The good news is that the Fed can recover slowly enough to keep raising interest rates in March.
The good news is that the Fed can accelerate quickly enough to raise interest rates in March.
The good news is that the Fed can taper up the interest rates fast enough to allow it to increase in March.
The good news is that the Fed can be able to reduce interest rates quick enough to raise them in March.
